In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns



from tqdm import tqdm
tqdm.pandas()

## Data Exploration

In [ ]:
bid_ask = pd.read_csv("../data/test/ETH.csv")
print(bid_ask.columns)

Index(['timestamp', 'mid_price', 'bid_price1', 'bid_volume1', 'bid_price2',
       'bid_volume2', 'bid_price3', 'bid_volume3', 'bid_price4', 'bid_volume4',
       'bid_price5', 'bid_volume5', 'ask_price1', 'ask_volume1', 'ask_price2',
       'ask_volume2', 'ask_price3', 'ask_volume3', 'ask_price4', 'ask_volume4',
       'ask_price5', 'ask_volume5', 'label'],
      dtype='object')


In [4]:
bid_ask["timestamp"] = pd.to_datetime(bid_ask["timestamp"], format = "%d-%m-%Y %H:%M")

ValueError: time data "2024-09-25 18:13:28" doesn't match format "%d-%m-%Y %H:%M", at position 0. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

In [143]:
print(f"Verifying time series data continuity:"
      f"\n{bid_ask["timestamp"].nunique() == ((bid_ask["timestamp"].iloc[-1] - bid_ask["timestamp"].iloc[0]).total_seconds() / 60 + 1)}"

      f"\n\nDistribution of tick counts per minute (number of seconds per minute): (expected to have top and bottom < 60, confirmed)"
      f"\n{bid_ask.groupby('timestamp')['timestamp'].count().value_counts()}"

      f"\n\nCount of NAN values across the dataset only for verification:"
      f"\n{bid_ask['mid_price'].isna().sum()}"
)

Verifying time series data continuity:
True

Distribution of tick counts per minute (number of seconds per minute): (expected to have top and bottom < 60, confirmed)
timestamp
60    4508
50       1
18       1
Name: count, dtype: int64

Count of NAN values across the dataset only for verification:
4596


So as evident from above, and with further exploratory analysis. It's determined that the dataset has full implied second resolution by index, albeit not labelled; has some missing values for its columns.

### Time resolution verification + labelling

In [144]:
bid_ask["timestamp"] = bid_ask["timestamp"].dt.floor(freq = "min")      # flooring the data to minute resolution

bid_ask["timestamp_second"] = bid_ask.groupby("timestamp").cumcount()

# handling edge case for the first minute - 
#  assumption: head of the missing data is missing, and the tail end is coninuous with the next minute
mask = bid_ask["timestamp"] == "2024-10-03 01:35:00" 
bid_ask.loc[ mask, "timestamp_second"] = bid_ask.loc[ mask, "timestamp_second"] + (60 - bid_ask.loc[mask, "timestamp"].count())

bid_ask["timestamp"] = bid_ask["timestamp"] + pd.to_timedelta(bid_ask["timestamp_second"], unit="s")
bid_ask.drop("timestamp_second", axis = 1, inplace = True)

In [148]:
bid_ask.head(5)

,timestamp,mid_price,bid_price1,bid_volume1,bid_price2,bid_volume2,bid_price3,bid_volume3,bid_price4,bid_volume4,...,ask_price1,ask_volume1,ask_price2,ask_volume2,ask_price3,ask_volume3,ask_price4,ask_volume4,ask_price5,ask_volume5
0,2024-10-03 01:35:10,2388.005,2388.0,433.6,2387.99,81.4,2387.96,10.8,2387.94,6.5,...,2388.01,313.0,2388.02,1.3,2388.07,0.1,2388.10,0.2,2388.11,14.9
1,2024-10-03 01:35:11,2388.005,2388.0,241.0,2387.99,81.4,2387.98,0.6,2387.94,6.5,...,2388.01,316.7,2388.02,14.3,2388.07,0.1,2388.10,24.0,2388.12,155.4
2,2024-10-03 01:35:12,2387.605,2387.6,22.6,2387.59,4.0,2387.57,0.4,2387.55,10.0,...,2387.61,1816.5,2387.62,13.0,2387.64,104.2,2387.65,20.3,2387.66,0.6
3,2024-10-03 01:35:13,2387.405,2387.4,246.0,2387.39,0.2,2387.36,62.9,2387.35,15.0,...,2387.41,769.7,2387.46,73.8,2387.47,208.0,2387.50,0.2,2387.52,38.4
4,2024-10-03 01:35:14,2387.005,2387.0,388.1,2386.99,30.0,2386.97,0.1,2386.96,0.4,...,2387.01,318.1,2387.02,1.3,2387.09,16.6,2387.10,25.5,2387.13,49.2


### Handling NAN values

In [150]:
# print(bid_ask.isna().sum()) # Entire rows are missing at a time

# Methodology used: Forward fill, at least for now
bid_ask = bid_ask.ffill(axis = 0)
print(f"Post forward fill, na values count in the dataset = {bid_ask.isna().sum().sum()}")

Post forward fill, na values count in the dataset = 0


In [149]:
bid_ask.head(5)

,timestamp,mid_price,bid_price1,bid_volume1,bid_price2,bid_volume2,bid_price3,bid_volume3,bid_price4,bid_volume4,...,ask_price1,ask_volume1,ask_price2,ask_volume2,ask_price3,ask_volume3,ask_price4,ask_volume4,ask_price5,ask_volume5
0,2024-10-03 01:35:10,2388.005,2388.0,433.6,2387.99,81.4,2387.96,10.8,2387.94,6.5,...,2388.01,313.0,2388.02,1.3,2388.07,0.1,2388.10,0.2,2388.11,14.9
1,2024-10-03 01:35:11,2388.005,2388.0,241.0,2387.99,81.4,2387.98,0.6,2387.94,6.5,...,2388.01,316.7,2388.02,14.3,2388.07,0.1,2388.10,24.0,2388.12,155.4
2,2024-10-03 01:35:12,2387.605,2387.6,22.6,2387.59,4.0,2387.57,0.4,2387.55,10.0,...,2387.61,1816.5,2387.62,13.0,2387.64,104.2,2387.65,20.3,2387.66,0.6
3,2024-10-03 01:35:13,2387.405,2387.4,246.0,2387.39,0.2,2387.36,62.9,2387.35,15.0,...,2387.41,769.7,2387.46,73.8,2387.47,208.0,2387.50,0.2,2387.52,38.4
4,2024-10-03 01:35:14,2387.005,2387.0,388.1,2386.99,30.0,2386.97,0.1,2386.96,0.4,...,2387.01,318.1,2387.02,1.3,2387.09,16.6,2387.10,25.5,2387.13,49.2


---